# 03 - Evaluation

Offline policy evaluation, regret analysis, cohort analysis, shock recovery, and final policy comparison.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluate import (
    cohort_analysis,
    compute_exploration_ratio,
    compute_regret,
    policy_comparison_table,
    shock_recovery_analysis,
)

In [ ]:
results_path = Path('../data/simulation_results.csv')
users_path = Path('../data/synthetic_users.csv')

if not results_path.exists():
    raise FileNotFoundError('Run src/simulate_run.py first to generate data/simulation_results.csv')

results_df = pd.read_csv(results_path)
users_df = pd.read_csv(users_path) if users_path.exists() else None
results_df.head()


In [ ]:
policy_frames = {
    policy: frame.copy()
    for policy, frame in results_df.groupby('policy')
}

# Use practical oracle as the baseline
oracle_df = policy_frames['oracle_practical']
regret_frames = []
for policy_name in ['thompson_sampling', 'ucb', 'epsilon_greedy']:
    regret_df = compute_regret(oracle_df, policy_frames[policy_name])
    regret_df['policy'] = policy_name
    regret_frames.append(regret_df)

all_regrets_df = pd.concat(regret_frames, ignore_index=True)
all_regrets_df.head()

In [ ]:
cohort_df = cohort_analysis(policy_frames['thompson_sampling'], ['risk_tier', 'income_bucket'])
cohort_df.head(10)


In [ ]:
exploration_ratio = compute_exploration_ratio(policy_frames['thompson_sampling'])
exploration_breakdown = compute_exploration_ratio.breakdown
print('Thompson exploration ratio (% non-keep):', round(exploration_ratio, 2))
print('Breakdown:', exploration_breakdown)

pre_shock_df = policy_frames['thompson_sampling'].query('month < 6')
post_shock_df = policy_frames['thompson_sampling'].query('month >= 6')
shock_metrics = shock_recovery_analysis(pre_shock_df, post_shock_df, n_months_post=6)
shock_metrics


In [ ]:
comparison_df = policy_comparison_table(policy_frames)
styled_comparison = comparison_df.style.format({
    'total_revenue_inr': '{:,.2f}',
    'default_rate_pct': '{:.2f}',
    'regret_vs_oracle_pct': '{:.2f}',
    'exploration_ratio_pct': '{:.2f}',
    'revenue_lift_vs_static_pct': '{:.2f}',
}).background_gradient(subset=['total_revenue_inr', 'revenue_lift_vs_static_pct'], cmap='YlGn')
display(styled_comparison)


In [ ]:
fig = px.line(
    all_regrets_df,
    x='month',
    y='regret',
    color='policy',
    markers=True,
    title='Regret Curves vs Oracle',
)
fig.update_layout(yaxis_title='Cumulative Regret (INR)', xaxis_title='Month')
fig.show()


In [ ]:
heatmap_df = cohort_analysis(policy_frames['thompson_sampling'], ['risk_tier', 'income_bucket'])
heatmap_matrix = heatmap_df.pivot(index='risk_tier', columns='income_bucket', values='avg_reward_per_user').fillna(0.0)
heatmap = px.imshow(
    heatmap_matrix,
    text_auto='.0f',
    aspect='auto',
    color_continuous_scale='Viridis',
    title='Thompson Cohort Heatmap: Risk Tier x Income Bucket -> Avg Reward',
)
heatmap.show()


In [ ]:
thompson_row = comparison_df.loc[comparison_df['policy'] == 'thompson_sampling'].iloc[0]
print('Target checks for Thompson Sampling:')
print('Revenue lift vs static > 30%:', thompson_row['revenue_lift_vs_static_pct'] > 30)
print('Default rate < 4%:', thompson_row['default_rate_pct'] < 4)
print(thompson_row[['revenue_lift_vs_static_pct', 'default_rate_pct']])
